# Unsloth Fine-tuning DeepSeek R1 Distilled Llama 8B

In this notebook, it will demonstrate how to finetune `DeepSeek-R1-Distill-Llama-8B` with Unsloth, using a medical dataset.

## Why do we need LLM fine-tuning?

Fine-tuning tailors the model to have a better performance for specific tasks, making it more effective and versatile in real-world applications. This process is essential for tailoring an existing model to a particular task or domain.

In [1]:
%%capture
!pip install unsloth
# Also get the latest nightly Unsloth!
# !pip uninstall unsloth -y && pip install --upgrade --no-cache-dir --no-deps git+https://github.com/unslothai/unsloth.git
!pip install bitsandbytes unsloth_zoo


## Choose a Base Model

1. Choose a model that aligns with your usecase
2. Assess your storage, compute capacity and dataset
3. Select a Model and Parameters
4. Choose Between Base and Instruct Models

In [2]:
!pip freeze >unsloth_requirement.txt

In [ ]:
from unsloth import FastLanguageModel
import torch
import os
max_seq_length = 2048 # Choose any! We auto support RoPE Scaling internally!
dtype = None # None for auto detection. Float16 for Tesla T4, V100, Bfloat16 for Ampere+
load_in_4bit = True # Use 4bit quantization to reduce memory usage. Can be False.

# 设置保存模型的路径
save_directory = "./saved_models/deepseek_r1"
os.makedirs(save_directory, exist_ok=True)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/DeepSeek-R1-Distill-Llama-8B",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
    # token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

# 保存模型权重到指定路径
model.save_pretrained(save_directory)
tokenizer.save_pretrained(save_directory)
print(f"Model weights saved to {save_directory}")

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.3.18: Fast Llama patching. Transformers: 4.49.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 7.5. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/5.96G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/236 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/53.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

In [ ]:
!du -h --max-depth=1 /root/.cache/huggingface/hub

36K	/root/.cache/huggingface/hub/models--unslothai--repeat
36K	/root/.cache/huggingface/hub/models--unslothai--colabpro
24K	/root/.cache/huggingface/hub/.locks
5.6G	/root/.cache/huggingface/hub/models--unsloth--deepseek-r1-distill-llama-8b-unsloth-bnb-4bit
36K	/root/.cache/huggingface/hub/models--unslothai--vram-24
36K	/root/.cache/huggingface/hub/models--unslothai--1
5.6G	/root/.cache/huggingface/hub


In [ ]:
!ls /root/.cache/huggingface/hub/models--unsloth--deepseek-r1-distill-llama-8b-unsloth-bnb-4bit/snapshots/5e0ac06dc3e90b3e84ce2c4b6bd3257974b1bb0a -l

total 4
lrwxrwxrwx 1 root root 52 Mar 19 15:02 config.json -> ../../blobs/cb322fcc15de3ea026d96a4bf0af57fca5310978
lrwxrwxrwx 1 root root 52 Mar 19 15:03 generation_config.json -> ../../blobs/846e0e5df0c5f053a21f9390ceec3eabc52d06b3
lrwxrwxrwx 1 root root 76 Mar 19 15:03 model.safetensors -> ../../blobs/e1a97f145b35eb6ed1844a3cc3a42b381780c3dc0248f796520a516fdc74dfeb


In [ ]:
!cat /root/.cache/huggingface/hub/version.txt

1

## Inference before fine-tuning

In [4]:
prompt_style = """下面是描述任务的说明，以及提供更多上下文的输入。写出合适的完成请求的回复。
回答之前，请仔细思考问题，一步一步地思考问题，确保回答合乎逻辑且准确。

### Instruction:
您是一位医学专家，在医学，药学，临床等方面拥有丰富的知识。
请回答以下医学问题。

### Question:
{}

### Response:
<think>{}"""

In [6]:
question = "一名 9 个月大的儿童一小时前出现癫痫发作，经检查发现其血清钙水平为 5 mg%。针对癫痫发作的根本原因，建议采取何种治疗方法？"


FastLanguageModel.for_inference(model) #必须切换到推理模式
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)#这一步是没有微调前进行一个推理
print(response[0].split("### Response:")[1])


<think>
嗯，我现在要回答一个关于儿童癫痫发作的医学问题。首先，问题描述是：一个9个月大的儿童一小时前出现癫痫发作，检查发现血清钙水平为5 mg%。问题是要建议治疗癫痫发作的根本原因。

首先，我需要理解癫痫发作的常见原因。癫痫通常是由于血糖、血脂、血液容量等代谢异常引起的，这些异常通常与新陈代谢紊乱有关。对于小婴儿来说，常见的原因包括低血糖、低血钙、低血钾、低血磷酸、低血蛋白等。

接下来，血清钙水平是5 mg%。正常的血钙浓度大约在8.5-10.5 mg%左右，所以5 mg%明显偏低，这可能是一个重要的因素。低血钙会导致神经系统过度兴奋，进而引发癫痫发作。因此，补钙应该是治疗的重要措施之一。

但是，补钙不能仅靠饮食调整，因为小婴儿的吸收和需求很高。直接补充钙剂可能会导致中毒，尤其是钙剂的剂量控制比较困难。因此，可能需要通过静脉补钙，但要注意不要过量，以免引起其他并发症。

另外，除了低血钙，还需要检查其他可能的因素，比如血糖、血脂、血蛋白等。低血糖也会导致癫痫发作，同样需要进行相应的处理。

所以，综合来看，治疗的根本原因可能是低血钙，而治疗方法应该是通过静脉补充钙剂，同时监测血钙水平，避免过量。同时，排除其他可能的代谢异常，进行全面的评估和治疗。

总结一下，治疗的根本原因是低血钙，建议通过静脉补钙，但要注意剂量控制，避免中毒，同时检查其他代谢指标，确保全面处理。
</think>

针对9个月大的儿童癫痫发作的根本原因，建议采取的治疗方法是通过静脉补充钙剂，同时监测血钙水平，避免过量。治疗的根本原因是低血钙。<｜end▁of▁sentence｜>


## Prepare Dataset

A medical dataset [https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/](https://huggingface.co/datasets/FreedomIntelligence/medical-o1-reasoning-SFT/) will be used to train the selected model.

In [62]:
train_prompt_style = """下面是描述任务的说明，以及提供更多上下文的输入。写出合适的完成请求的回复。
回答之前，请仔细思考问题，一步一步地思考问题，确保回答合乎逻辑且准确。
请适度思考，不要过度思考

### Instruction:
您是一位医学专家，在医学，药学，临床等方面拥有丰富的知识。
请回答以下医学问题。

### Question:
{}

### Response:
<think>{}"""


### Important Notice

It's crucial to add the EOS (End of Sequence) token at the end of each training dataset entry, otherwise you may encounter infinite generations.

In [63]:
EOS_TOKEN = tokenizer.eos_token  # Must add EOS_TOKEN


def formatting_prompts_func(examples):
    inputs = examples["question"]
    cots = examples["reasoning (reasoning_content)"]
    outputs = examples["response (content)"]
    texts = []
    for input, cot, output in zip(inputs, cots, outputs):
        text = train_prompt_style.format(input, cot, output) + EOS_TOKEN
        texts.append(text)
    return {
        "text": texts,
    }

In [64]:
from datasets import load_dataset
dataset = load_dataset("FreedomIntelligence/Medical-R1-Distill-Data-Chinese", 'zh', split = "train[:30%]", trust_remote_code=True)
print(dataset.column_names)

['question', 'reasoning (reasoning_content)', 'response (content)']


For `Ollama` and `llama.cpp` to function like a custom `ChatGPT` Chatbot, we must only have 2 columns - an `instruction` and an `output` column. We need to transform the dataset into proper structure.

In [65]:
dataset = dataset.map(formatting_prompts_func, batched = True)
dataset["text"][0]

Map:   0%|          | 0/5100 [00:00<?, ? examples/s]

'下面是描述任务的说明，以及提供更多上下文的输入。写出合适的完成请求的回复。\n回答之前，请仔细思考问题，一步一步地思考问题，确保回答合乎逻辑且准确。\n请适度思考，不要过度思考\n\n### Instruction:\n您是一位医学专家，在医学，药学，临床等方面拥有丰富的知识。\n请回答以下医学问题。\n\n### Question:\n根据患者59岁女性，出现胸闷、憋喘症状和相关的体格检查结果，其中X线显示双侧胸腔积液，胸水检查结果显示血性积液且有异常细胞，痰细胞学检查提示癌细胞体积小、圆形或卵圆形、胞质少，核畸形并深染，结合这些检查结果，患者最可能的诊断是什么？\n\n### Response:\n<think>嗯，好的，我现在要解决这个病例的问题。首先，患者是59岁女性，主诉是胸闷和憋喘。体格检查发现有双侧胸腔积液，X线确认了这一点。胸水检查显示是血性积液，并且有异常细胞。痰细胞学检查发现癌细胞体积小，呈圆形或卵圆形，胞质少，核畸形且深染。我需要根据这些信息来推断最可能的诊断。\n\n首先，我应该回顾一下胸腔积液的常见原因。胸腔积液可能由多种原因引起，包括充血性心力衰竭、感染（如肺炎旁积液）、恶性肿瘤、肺栓塞、结核等。但这里胸水是血性的，并且发现了异常细胞，所以恶性肿瘤的可能性更大。血性胸腔积液常见于恶性肿瘤，尤其是肺癌转移至胸膜，或者间皮瘤，或者其他转移癌，比如乳腺癌、卵巢癌等转移到胸膜。但患者是女性，59岁，需要考虑原发肺癌的可能。\n\n接下来，痰细胞学检查的结果提示癌细胞体积小、圆形或卵圆形，胞质少，核畸形深染。这些特征可能指向某种特定类型的肺癌。肺癌的主要类型有小细胞癌、鳞癌、腺癌和大细胞癌。其中，小细胞肺癌的细胞学特征通常是细胞小，呈圆形或卵圆形，胞质少，核染色深，核畸形可能不太明显，但可能有挤压现象。而腺癌的细胞通常较大，胞质可能较多，形成腺泡状结构，核可能较大，但这里的描述是体积小，所以可能更倾向于小细胞癌。\n\n不过，小细胞肺癌通常发生在中央型，容易早期转移，常见于吸烟者。患者是否有吸烟史呢？但题目中没有提到，所以不能确定。但痰中发现癌细胞，说明肿瘤可能位于支气管内，容易脱落到痰中，这可能更符合中央型肺癌，如小细胞癌或鳞癌。但鳞癌的细胞学特征通常是较大的细胞，胞质较多，角化现象，核深染但可能有更明显的核仁，而这里的描述是细胞小，所以可能更

## Train the model
Now let's use Huggingface TRL's `SFTTrainer`.

In [66]:
print(dataset['response (content)'][0])

根据患者症状、影像学及实验室检查结果，最可能的诊断是 **小细胞肺癌（Small Cell Lung Cancer, SCLC）伴胸膜转移**。以下是关键分析步骤：

---

### **关键诊断依据**
1. **临床表现与影像学**  
   - **胸闷、憋喘**：提示胸腔积液导致肺组织受压，影响通气功能。  
   - **双侧胸腔积液（血性）**：血性胸腔积液高度提示恶性肿瘤，尤其是胸膜转移（如肺癌、乳腺癌等）。  

2. **胸水及痰细胞学特征**  
   - **胸水发现异常细胞**：支持恶性积液（如转移性癌细胞或间皮瘤）。  
   - **痰细胞学特征**：  
     - **细胞体积小、圆形/卵圆形**：符合小细胞肺癌的“燕麦细胞”形态。  
     - **胞质少、核畸形深染**：核深染、高核质比是小细胞肺癌的典型特征，可能与神经内分泌分化相关。  

3. **病理生理联系**  
   - 小细胞肺癌恶性度高，早期易转移（如胸膜、肝、脑），常合并癌性胸腔积液。  
   - 中央型肺癌（如小细胞癌）易侵犯大气道，癌细胞易脱落至痰液中，与痰检结果一致。

---

### **鉴别诊断**
1. **肺腺癌**  
   - 痰细胞学多表现为大细胞、胞质丰富、核仁明显，与本例不符。  
   - 胸膜转移常见，但细胞形态不同。  

2. **间皮瘤**  
   - 胸水细胞学可见间皮细胞增生（胞质丰富、核大、核仁明显），但本例细胞特征更符合小细胞癌。  

3. **其他转移性癌（如乳腺癌、卵巢癌）**  
   - 需结合原发灶病史及免疫组化（如TTF-1阴性、GATA3阳性等）排除，但痰检发现癌细胞更支持肺部原发。

---

### **进一步检查建议**
1. **胸部增强CT**：明确肺部原发肿瘤位置及胸膜受累情况。  
2. **支气管镜活检或胸膜活检**：病理确诊（免疫组化如CD56、Syn、CgA等神经内分泌标记物阳性支持小细胞癌）。  
3. **全身分期检查（如PET-CT、脑MRI）**：评估转移范围，指导治疗。

---

### **总结**
患者表现为血性胸腔积液、痰中检出典型小细胞癌特征的癌细胞，结合小细胞肺癌的高侵袭性和胸膜转移倾向，**小细胞肺癌伴胸膜转移**是最可能的诊断。需尽快病理确诊并启

In [67]:
FastLanguageModel.for_training(model)

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(128256, 4096, padding_idx=128004)
        (layers): ModuleList(
          (0): LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear

In [68]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, # Supports any, but = 0 is optimized
    bias = "none",    # Supports any, but = "none" is optimized
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing = "unsloth", # True or "unsloth" for very long context
    random_state = 3407,
    use_rslora = False,  # We support rank stabilized LoRA
    loftq_config = None, # And LoftQ
)

Unsloth: Already have LoRA adapters! We shall skip this step.


In [69]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,#启动几个进程去dataset中加载数据
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 600,
        # num_train_epochs = 1, # For longer training runs!
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc，none是不用任何报告插件
    ),
)

Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/5100 [00:00<?, ? examples/s]

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,100 | Num Epochs = 1 | Total steps = 600
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040/8,000,000,000 (0.52% trained)


Step,Training Loss
1,1.006700
2,1.072200
3,1.057500
4,0.937500
5,1.012000
6,1.013300


## Inference after fine-tuning

Let's inference with same question again and see the difference.

In [ ]:
print(question)

In [ ]:
FastLanguageModel.for_inference(model)  # Unsloth has 2x faster inference! 验证微调后的模型可以推理
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1200,
    use_cache=True,
)
response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])

In [ ]:
#为了加快测试速度，所以在最后50个样本做测试
test_dataset = load_dataset("FreedomIntelligence/Medical-R1-Distill-Data-Chinese", 'zh', split = "train[-20:]", trust_remote_code=True)
test_questions = test_dataset["question"]
test_references = test_dataset["response (content)"]  # 参考答案
print(test_questions[0])
print('-'*50)
test_references[0]

In [ ]:

# 生成模型的预测结果,可以考虑去掉<Think>标签内的内容，还有最后的<｜end▁of▁sentence｜>
model_predictions = []
for question in test_questions[:1]:
    inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")
    outputs = model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_new_tokens=1200,
        use_cache=True,
    )
    ok = tokenizer.batch_decode(outputs)
    print(ok)
    response = tokenizer.batch_decode(outputs)[0].split("### Response:")[1]
    print(response)
    response = response.split("</think>")[1].strip()
    response = response.replace("<｜end▁of▁sentence｜>", "")
    model_predictions.append(response)


In [61]:
if '</think>' in ok:
  print('yes')
else:
  print('no')

no


In [50]:
print(response)


<think>嗯，我现在要解决这个问题：一位42岁的女性肝硬化患者入院时发生了昏迷，血钾浓度低至2.4mmol/L。现在需要选择纠正低钾血症的药物。首先，我需要回忆一下低钾血症的处理原则，以及肝硬化患者的特殊情况。

首先，低钾血症的治疗通常有几个选择。常见的药物包括钾补充剂，比如钾盐、钾糖醇、氯化钾等。另外，有些药物可能会影响钾的吸收或排泄，比如丙磺舒、吡磷酰胺等，但这些可能不是直接补充钾的，而是通过改变肾脏的钾代谢来提高血钾浓度。不过题目问的是纠正低钾血症，应该是补充钾的药物。

接下来，肝硬化患者的特殊情况。肝硬化导致肝脏功能受损，无法有效地排出水分和钾，所以肝硬化患者容易发生水钾失衡。尤其是如果有肾功能不全的话，排出钾的能力会更低，所以血钾浓度会更低。这种情况下，补充钾需要考虑肾功能的情况。

患者的血钾浓度是2.4mmol/L，这明显低于正常范围（通常正常男性的血钾浓度在3.5-6.5mmol/L，女性可能稍低一些）。这种情况下，低钾血症是严重的，可能需要紧急处理。因为低钾血症可能导致心动过缓、肌肉抽搐、甚至昏迷，所以需要及时纠正。

对于肝硬化患者，如果没有肾功能不全的情况下，补钾的选择可能包括钾盐、钾糖醇、钾溶液（比如钾葡萄糖酸钠），或者氯化钾（尤其是如果患者不能口服的话）。不过，如果肾功能不全的话，可能需要使用氯化钾，因为氯化钾可以被肾脏排出，而其他钾盐可能在肾脏中被保留，导致钾浓度进一步升高，这可能对肾功能不全的患者不利。所以，如果肝硬化患者同时有肾功能不全，应该优先使用氯化钾补充钾，避免钾盐的积累。

不过题目中提到的是肝硬化患者，可能肾功能是否正常呢？如果没有特别说明肾功能不全的话，可能需要考虑两种情况。但通常肝硬化患者如果没有肾功能不全的话，补钾可能需要用氯化钾，因为氯化钾在肾脏中被大量排出，而其他钾盐可能在肾脏中被保留，导致钾浓度升高，但如果肾功能正常的话，可能氯化钾更适合。不过，如果肾功能不全的话，氯化钾会被肾脏排出，可能导致低钾血症，反而需要补充钾盐，所以这个时候可能需要使用钾盐，但需要调整剂量，避免钾盐的积累。

不过题目中并没有提到肾功能是否正常，所以可能需要假设肾功能正常的情况下，或者是否存在肾功能不全的情况呢？比如肝硬化患者如果没有肾功能不全的话，可能肾功能正常，所以补钾可以使用氯化钾，但如果肾功能不全的话，可能需要钾盐。但通

In [ ]:
model_predictions[0]

In [ ]:
len(model_predictions)

In [ ]:
test_references[1]

In [ ]:
!pip install nltk rouge

In [ ]:
from nltk.translate.bleu_score import sentence_bleu
#因为推理太慢了，所以推理了34个后打断了，只对已经推理的34个样本计算了bleu指标
bleu_scores = []
for pred, ref in zip(model_predictions[0:31], test_references[0:31]):
    # 将参考答案和预测结果分词
    ref_tokens = [ref.split()]
    pred_tokens = pred.split()
    # 计算 BLEU 分数
    bleu_score = sentence_bleu(ref_tokens, pred_tokens,weights=(1,0,0,0))
    bleu_scores.append(bleu_score)

# 计算平均 BLEU 分数
average_bleu = sum(bleu_scores) / len(bleu_scores)
print(f"Average BLEU Score: {average_bleu}")


下面的部分可以不做，不用上传到抱抱脸上

## Upload Model to HuggingFace

Now, let's save our finetuned model and upload it to HuggingFace.

### Save the fine-tuned model to GGUF format

Choose the llama.cpp's GGUF format we prefer by setting the corresponding `if` to `True`.

In [ ]:
# from google.colab import userdata

# HUGGINGFACE_TOKEN = userdata.get('HUGGINGFACE_TOKEN')

In [ ]:
# # Save to 8bit Q8_0
# if True: model.save_pretrained_gguf("model", tokenizer,)

# # Save to 16bit GGUF
# if False: model.save_pretrained_gguf("model_f16", tokenizer, quantization_method = "f16")

# # Save to q4_k_m GGUF
# if False: model.save_pretrained_gguf("model", tokenizer, quantization_method = "q4_k_m")

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.
Unsloth: Kaggle/Colab has limited disk space. We need to delete the downloaded
model which will save 4-16GB of disk space, allowing you to save on Kaggle/Colab.
Unsloth: Will remove a cached repo with size 6.0G


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 34.71 out of 52.96 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


 91%|█████████ | 29/32 [00:01<00:00, 18.35it/s]
We will save to Disk and not RAM now.
100%|██████████| 32/32 [00:02<00:00, 12.46it/s]


Unsloth: Saving tokenizer... Done.
Done.


Unsloth: Converting llama model. Can use fast conversion = False.


==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: CMAKE detected. Finalizing some steps for installation.
Unsloth: [1] Converting model at model into q8_0 GGUF format.
The output location will be /content/model/unsloth.Q8_0.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: model
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part 'model-00001-of-00004.safetensors

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Unsloth: Conversion completed! Output location: /content/model/unsloth.Q8_0.gguf


### Push the model to HuggingFace

Create a model type repository for your model if you haven't done so.

In [ ]:
# from huggingface_hub import create_repo
# create_repo("wyang14/medical-model", token=HUGGINGFACE_TOKEN, exist_ok=True)

RepoUrl('https://huggingface.co/wyang14/medical-model', endpoint='https://huggingface.co', repo_type='model', repo_id='wyang14/medical-model')

In [ ]:
# model.push_to_hub_gguf("wyang14/medical-model", tokenizer, token = HUGGINGFACE_TOKEN)

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging 4bit and LoRA weights to 16bit...
Unsloth: Will use up to 33.04 out of 52.96 RAM for saving.
Unsloth: Saving model... This might take 5 minutes ...


100%|██████████| 32/32 [00:02<00:00, 13.40it/s]


Unsloth: Saving tokenizer... Done.
Done.
==((====))==  Unsloth: Conversion from QLoRA to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF 16bits might take 3 minutes.
\        /    [2] Converting GGUF 16bits to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: [1] Converting model at wyang14/medical-model into q8_0 GGUF format.
The output location will be /content/wyang14/medical-model/unsloth.Q8_0.gguf
This might take 3 minutes...
INFO:hf-to-gguf:Loading model: medical-model
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:rope_freqs.weight,           torch.float32 --> F32, shape = {64}
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: loading model part 'model-00001-of-0

  0%|          | 0/1 [00:00<?, ?it/s]

unsloth.Q8_0.gguf:   0%|          | 0.00/8.54G [00:00<?, ?B/s]

Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### We removed it in GGUF's chat template for you.


Saved GGUF to https://huggingface.co/wyang14/medical-model


### Ollama run HuggingFace model

```bash
ollama run hf.co/{username}/{repository}:{quantization}
```

### Ollama inference

```bash
curl http://localhost:11434/api/chat -d '{ \
  "model": "", \
  "messages": [ \
    { "role": "user", "content": "一个患有急性阑尾炎的病人已经发病5天，腹痛稍有减轻但仍然发热，在体检时发现右下腹有压痛的包块，此时应如何处理？" } \
  }'
```